# Ungraded Lab: U-Net for Image Segmentation (PyTorch)

This notebook illustrates how to build a [UNet](https://arxiv.org/abs/1505.04597) for semantic image segmentation. This architecture is also a fully convolutional network and is similar to the model you just built in the previous lesson. A key difference is the use of skip connections from the encoder to the decoder. You will see how this is implemented later as you build each part of the network.

At the end of this lab, you will be able to use the UNet to output segmentation masks that show which pixels of an input image are part of the background, foreground, and outline.

> This notebook is a PyTorch port of the original TensorFlow lab. The dataset comes from `torchvision.datasets.OxfordIIITPet` instead of TensorFlow Datasets, the blocks are `nn.Module`s instead of Keras functional layers, and `model.fit` becomes an explicit training loop.

## Download the Oxford-IIIT Pets dataset

You will be training the model on the [Oxford Pets - IIT dataset](https://www.robots.ox.ac.uk/~vgg/data/pets/). This contains pet images, their classes, segmentation masks and head region-of-interest. You will only use the images and segmentation masks in this lab.

torchvision can download it for you. Ask for `target_types="segmentation"` so each item is an (image, mask) pair.

_Note: the download is roughly 800 MB and only happens the first time._

## Imports

In [ ]:
import platform

import numpy as np
import matplotlib.pyplot as plt
import PIL.Image

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import OxfordIIITPet
from torchvision.transforms import functional as TF
from torchinfo import summary

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

# DataLoader workers on macOS start with "spawn", which cannot see classes defined in a notebook.
MP_CONTEXT = "fork" if platform.system() == "Darwin" else None

In [ ]:
# download the dataset and get its two splits
raw_train = OxfordIIITPet(root="data", split="trainval", target_types="segmentation", download=True)
raw_test = OxfordIIITPet(root="data", split="test", target_types="segmentation", download=True)

print(f"train: {len(raw_train)} images")
print(f"test:  {len(raw_test)} images")

## Prepare the Dataset

You will now prepare the train and test sets. The following utilities preprocess the data. These include:

* simple augmentation by flipping the image
* normalizing the pixel values  
* resizing the images

Another preprocessing step is to adjust the segmentation mask's pixel values. The `README` in the [annotations](https://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz) folder of the dataset mentions that the pixels in the segmentation mask are labeled as such:

| Label            | Class Name     |
| -------------    | -------------  |
| 1                | foreground     |
| 2                | background     |
| 3                | Not Classified |

<br>
<br>

For convenience, let's subtract `1` from these values and we will interpret these as `{'pet', 'background', 'outline'}`:

| Label            | Class Name     |
| -------------    | -------------  |
| 0                | pet            |
| 1                | background     |
| 2                | outline        |

One PyTorch specific note: the mask must be resized with **nearest neighbour** interpolation. Anything that averages neighbouring pixels would invent class ids that do not exist.

In [ ]:
# Preprocessing Utilities

def random_flip(input_image, input_mask):
    '''
    Does a random horizontal flip of the image and its mask.

    Args:
      input_image (tensor) -- image, shape (3, H, W)
      input_mask (tensor) -- label map, shape (H, W)

    Returns:
      (tensor, tensor) -- the pair, flipped together or left as they were
    '''
    if torch.rand(()) > 0.5:
        input_image = TF.hflip(input_image)
        input_mask = TF.hflip(input_mask)

    return input_image, input_mask


def normalize(input_image, input_mask):
    '''
    Normalizes the input image pixel values to be from [0, 1].
    Subtracts 1 from the mask labels to have a range from [0, 2].

    Args:
      input_image (tensor) -- uint8 image, shape (3, H, W)
      input_mask (tensor) -- raw mask with labels 1, 2 and 3

    Returns:
      (tensor, tensor) -- float image in [0, 1] and the label map in {0, 1, 2}
    '''
    input_image = input_image.float() / 255.0
    input_mask = input_mask - 1
    return input_image, input_mask


def load_image(image, mask, train):
    '''
    Resizes, normalizes, and optionally flips one image and mask pair.

    Args:
      image (PIL.Image) -- the raw pet photo
      mask (PIL.Image) -- the raw segmentation mask
      train (bool) -- True applies the random flip used for the training split

    Returns:
      (tensor, tensor) -- image of shape (3, 128, 128) and label map of shape (128, 128)
    '''
    input_image = image.convert("RGB").resize((128, 128), PIL.Image.NEAREST)
    input_mask = mask.resize((128, 128), PIL.Image.NEAREST)

    input_image = torch.from_numpy(np.array(input_image)).permute(2, 0, 1)
    input_mask = torch.from_numpy(np.array(input_mask)).long()

    if train:
        input_image, input_mask = random_flip(input_image, input_mask)
    input_image, input_mask = normalize(input_image, input_mask)

    return input_image, input_mask


class PetsSegmentation(Dataset):
    '''Oxford-IIIT Pet images paired with their three-class segmentation masks.'''

    def __init__(self, raw, train):
        '''
        Stores the split this dataset serves and whether to augment it.

        Args:
          raw (OxfordIIITPet) -- the downloaded split, yielding (image, mask) PIL pairs
          train (bool) -- True enables the random flip augmentation
        '''
        self.raw = raw
        self.train = train

    def __len__(self):
        '''
        Reports how many pairs this split holds.

        Returns:
          int -- number of image and mask pairs in this split
        '''
        return len(self.raw)

    def __getitem__(self, idx):
        '''
        Loads and preprocesses pair `idx`.

        Args:
          idx (int) -- index of the pair to fetch

        Returns:
          (tensor, tensor) -- preprocessed image (3, 128, 128) and label map (128, 128)
        '''
        image, mask = self.raw[idx]
        return load_image(image, mask, self.train)

You can now wrap the two splits and group them into batches. In TensorFlow the shuffling, batching and prefetching were chained onto the dataset; in PyTorch the `DataLoader` handles all three.

In [ ]:
BATCH_SIZE = 64

train = PetsSegmentation(raw_train, train=True)
test = PetsSegmentation(raw_test, train=False)

# shuffle and group the train set into batches, prefetching with worker processes
train_dataset = DataLoader(train, batch_size=BATCH_SIZE, shuffle=True, num_workers=4,
                           multiprocessing_context=MP_CONTEXT, persistent_workers=True)

# group the test set into batches
test_dataset = DataLoader(test, batch_size=BATCH_SIZE, num_workers=4,
                          multiprocessing_context=MP_CONTEXT, persistent_workers=True)

Let's define a few more utilities to help us visualize our data and metrics.

In [ ]:
# class list of the mask pixels
class_names = ['pet', 'background', 'outline']


def to_displayable(item):
    '''
    Turns an image or label map tensor into something matplotlib can show.

    Args:
      item (tensor) -- either an image (3, H, W) in [0, 1] or a label map (H, W)

    Returns:
      array -- (H, W, 3) float image, or (H, W) label map
    '''
    item = item.detach().cpu() if torch.is_tensor(item) else torch.as_tensor(item)
    if item.ndim == 3 and item.shape[0] == 3:
        return item.permute(1, 2, 0).numpy()
    return item.squeeze().numpy()


def display(display_list, titles=[], display_string=None):
    '''
    Displays a list of images and masks side by side.

    Args:
      display_list (list) -- image and label map tensors to show
      titles (list of str) -- a title per entry
      display_string (string) -- optional text placed under the second entry
    '''
    plt.figure(figsize=(15, 15))

    for i in range(len(display_list)):
        plt.subplot(1, len(display_list), i + 1)
        if i < len(titles):
            plt.title(titles[i])
        plt.xticks([])
        plt.yticks([])
        if display_string and i == 1:
            plt.xlabel(display_string, fontsize=12)
        plt.imshow(to_displayable(display_list[i]))

    plt.show()


def display_with_metrics(display_list, iou_list, dice_score_list):
    '''
    Displays a list of images and masks, overlaying the IOU and Dice scores.

    Args:
      display_list (list) -- image, predicted mask and true mask
      iou_list (list of float) -- IOU per class
      dice_score_list (list of float) -- Dice score per class
    '''
    metrics_by_id = [(idx, iou, dice_score)
                     for idx, (iou, dice_score) in enumerate(zip(iou_list, dice_score_list)) if iou > 0.0]
    metrics_by_id.sort(key=lambda tup: tup[1], reverse=True)  # sorts in place

    display_string_list = ["{}: IOU: {} Dice Score: {}".format(class_names[idx], iou, dice_score)
                           for idx, iou, dice_score in metrics_by_id]
    display_string = "\n\n".join(display_string_list)

    display(display_list, ["Image", "Predicted Mask", "True Mask"], display_string=display_string)


def show_image_from_dataset(dataset):
    '''
    Displays the first image and its mask from a dataset.

    Args:
      dataset (Dataset) -- dataset yielding (image, mask) pairs
    '''
    sample_image, sample_mask = dataset[0]
    display([sample_image, sample_mask], titles=["Image", "True Mask"])


def plot_metrics(history, metric_name, title, ylim=5):
    '''
    Plots a training metric and its validation counterpart against the epoch number.

    Args:
      history (dict) -- metric name to list of per-epoch values
      metric_name (string) -- key to plot, for example 'loss'
      title (string) -- title for the figure
      ylim (float) -- upper limit of the y axis
    '''
    plt.title(title)
    plt.ylim(0, ylim)
    plt.plot(history[metric_name], color='blue', label=metric_name)
    plt.plot(history['val_' + metric_name], color='green', label='val_' + metric_name)
    plt.legend()

Finally, you can take a look at an image example and its corresponding mask from the dataset.

In [ ]:
# display an image from the train set
show_image_from_dataset(train)

# display an image from the test set
show_image_from_dataset(test)

## Define the model

With the dataset prepared, you can now build the UNet. A UNet consists of an encoder (downsampler) and decoder (upsampler) with a bottleneck in between. Skip connections concatenate encoder block outputs to each stage of the decoder. Let's see how to implement these starting with the encoder.

### Encoder

Like the FCN model you built in the previous lesson, the encoder here will have repeating blocks so it's best to create modules for them to keep the code modular. These encoder blocks will contain two convolution layers activated by ReLU, followed by a max pooling and dropout layer. Each stage will have an increasing number of filters and the dimensionality of the features will reduce because of the pooling layer.

The encoder utilities are three modules:

* `Conv2dBlock` - two convolution layers with ReLU activations
* `EncoderBlock` - pooling and dropout on top of a conv block. Recall that in UNet you need to save the output of the convolution layers at each block, so this returns two values (the conv block output and the pooled and dropped output)
* `Encoder` - the entire encoder. This returns the output of the last encoder block as well as the outputs of the previous conv blocks, which get concatenated into the decoder later.

PyTorch convolutions need their input channel count, so each block takes `in_channels`. Keras' `he_normal` initializer is `nn.init.kaiming_normal_`.

In [ ]:
# Encoder Utilities

class Conv2dBlock(nn.Module):
    '''Two convolution layers, each followed by a ReLU.'''

    def __init__(self, in_channels, n_filters, kernel_size=3):
        '''
        Builds the two convolutions and their ReLU activations.

        Args:
          in_channels (int) -- channels of the input tensor
          n_filters (int) -- number of filters for both convolutions
          kernel_size (int) -- kernel size for the convolutions
        '''
        super().__init__()
        layers = []
        channels = in_channels
        for _ in range(2):
            conv = nn.Conv2d(channels, n_filters, kernel_size=kernel_size, padding='same')
            nn.init.kaiming_normal_(conv.weight, nonlinearity='relu')   # Keras' he_normal
            nn.init.zeros_(conv.bias)
            layers += [conv, nn.ReLU()]
            channels = n_filters
        self.block = nn.Sequential(*layers)

    def forward(self, input_tensor):
        '''
        Runs the input through both convolutions.

        Args:
          input_tensor (tensor) -- input features, shape (N, in_channels, H, W)

        Returns:
          tensor -- output features, shape (N, n_filters, H, W)
        '''
        return self.block(input_tensor)


class EncoderBlock(nn.Module):
    '''A conv block followed by max pooling and dropout.'''

    def __init__(self, in_channels, n_filters=64, pool_size=2, dropout=0.3):
        '''
        Builds the conv block together with its pooling and dropout.

        Args:
          in_channels (int) -- channels of the input tensor
          n_filters (int) -- number of filters for the conv block
          pool_size (int) -- size of the pooling window
          dropout (float) -- between 0 and 1, rate of the dropout layer
        '''
        super().__init__()
        self.conv = Conv2dBlock(in_channels, n_filters)
        self.pool = nn.MaxPool2d(pool_size)
        self.drop = nn.Dropout(dropout)

    def forward(self, inputs):
        '''
        Convolves the input, then pools and drops it for the next stage.

        Args:
          inputs (tensor) -- input features

        Returns:
          (tensor, tensor) -- f, the conv block features kept for the skip connection,
          and p, the pooled features with dropout applied
        '''
        f = self.conv(inputs)
        p = self.drop(self.pool(f))
        return f, p


class Encoder(nn.Module):
    '''The downsampling path, four encoder blocks with increasing filters.'''

    def __init__(self):
        '''Builds the four encoder blocks with 64, 128, 256 and 512 filters.'''
        super().__init__()
        self.block1 = EncoderBlock(3, n_filters=64, pool_size=2, dropout=0.3)
        self.block2 = EncoderBlock(64, n_filters=128, pool_size=2, dropout=0.3)
        self.block3 = EncoderBlock(128, n_filters=256, pool_size=2, dropout=0.3)
        self.block4 = EncoderBlock(256, n_filters=512, pool_size=2, dropout=0.3)

    def forward(self, inputs):
        '''
        Runs the image down through the four encoder blocks.

        Args:
          inputs (tensor) -- batch of input images, shape (N, 3, 128, 128)

        Returns:
          (tensor, tuple) -- p4, the pooled output of the last encoder block (N, 512, 8, 8),
          and (f1, f2, f3, f4), the conv features of every block for the skip connections
        '''
        f1, p1 = self.block1(inputs)
        f2, p2 = self.block2(p1)
        f3, p3 = self.block3(p2)
        f4, p4 = self.block4(p3)

        return p4, (f1, f2, f3, f4)

### Bottleneck

A bottleneck follows the encoder block and is used to extract more features. This does not have a pooling layer so the dimensionality remains the same. You can use the `Conv2dBlock` defined earlier to implement this.

In [ ]:
def bottleneck(in_channels=512, n_filters=1024):
    '''
    Defines the bottleneck convolutions that extract more features before upsampling.

    Args:
      in_channels (int) -- channels coming out of the encoder
      n_filters (int) -- number of filters for the bottleneck convolutions

    Returns:
      nn.Module -- a conv block that keeps the spatial size unchanged
    '''
    return Conv2dBlock(in_channels, n_filters=n_filters)

### Decoder

Finally, we have the decoder which upsamples the features back to the original image size. At each upsampling level, you take the output of the corresponding encoder block and concatenate it before feeding to the next decoder block.

Keras' `Conv2DTranspose(..., strides=2, padding='same')` doubles the spatial size. The PyTorch equivalent is `ConvTranspose2d(..., stride=2, padding=1, output_padding=1)`.

In [ ]:
# Decoder Utilities

class DecoderBlock(nn.Module):
    '''One decoder block of the UNet: upsample, concatenate the skip features, then convolve.'''

    def __init__(self, in_channels, skip_channels, n_filters=64, kernel_size=3, strides=2, dropout=0.3):
        '''
        Builds the transposed convolution, dropout and conv block of one stage.

        Args:
          in_channels (int) -- channels of the incoming features
          skip_channels (int) -- channels of the encoder features being concatenated
          n_filters (int) -- number of filters
          kernel_size (int) -- kernel size for the deconvolution
          strides (int) -- stride for the deconvolution, 2 doubles the spatial size
          dropout (float) -- between 0 and 1, rate of the dropout layer
        '''
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, n_filters, kernel_size,
                                     stride=strides, padding=1, output_padding=strides - 1)
        self.drop = nn.Dropout(dropout)
        self.conv = Conv2dBlock(n_filters + skip_channels, n_filters, kernel_size=3)

    def forward(self, inputs, conv_output):
        '''
        Upsamples the input and merges it with the matching encoder features.

        Args:
          inputs (tensor) -- features from the previous decoder stage
          conv_output (tensor) -- matching features from the encoder

        Returns:
          tensor -- output features of the decoder block, at twice the input resolution
        '''
        u = self.up(inputs)
        c = torch.cat([u, conv_output], dim=1)   # Keras concatenates on the channel axis too
        c = self.drop(c)
        c = self.conv(c)
        return c


class Decoder(nn.Module):
    '''The upsampling path, four decoder blocks plus the final classifying convolution.'''

    def __init__(self, output_channels):
        '''
        Builds the four decoder blocks and the final classifying convolution.

        Args:
          output_channels (int) -- number of classes in the label map
        '''
        super().__init__()
        self.block6 = DecoderBlock(1024, 512, n_filters=512, kernel_size=3, strides=2, dropout=0.3)
        self.block7 = DecoderBlock(512, 256, n_filters=256, kernel_size=3, strides=2, dropout=0.3)
        self.block8 = DecoderBlock(256, 128, n_filters=128, kernel_size=3, strides=2, dropout=0.3)
        self.block9 = DecoderBlock(128, 64, n_filters=64, kernel_size=3, strides=2, dropout=0.3)
        self.outputs = nn.Conv2d(64, output_channels, kernel_size=1)

    def forward(self, inputs, convs):
        '''
        Upsamples the bottleneck back to full resolution using the skip connections.

        Args:
          inputs (tensor) -- the bottleneck features, shape (N, 1024, 8, 8)
          convs (tuple) -- the encoder features (f1, f2, f3, f4)

        Returns:
          tensor -- per-pixel class scores, shape (N, output_channels, 128, 128).
          These are logits; the softmax is applied by the loss during training and
          explicitly when you want probabilities.
        '''
        f1, f2, f3, f4 = convs

        c6 = self.block6(inputs, f4)
        c7 = self.block7(c6, f3)
        c8 = self.block8(c7, f2)
        c9 = self.block9(c8, f1)

        return self.outputs(c9)

### Putting it all together

You can finally build the UNet by chaining the encoder, bottleneck, and decoder. You will specify the number of output channels and in this particular set, that would be `3`. That is because there are three possible labels for each pixel: 'pet', 'background', and 'outline'.

In [ ]:
OUTPUT_CHANNELS = 3


class UNet(nn.Module):
    '''UNet built from the encoder, the bottleneck and the decoder.'''

    def __init__(self, output_channels=OUTPUT_CHANNELS):
        '''
        Builds the encoder, the bottleneck and the decoder.

        Args:
          output_channels (int) -- number of classes in the label map
        '''
        super().__init__()
        self.encoder = Encoder()
        self.bottleneck = bottleneck(512, 1024)
        self.decoder = Decoder(output_channels)

    def forward(self, inputs):
        '''
        Runs the image down the encoder, through the bottleneck, then back up the decoder.

        Args:
          inputs (tensor) -- batch of images, shape (N, 3, 128, 128)

        Returns:
          tensor -- per-pixel class scores, shape (N, output_channels, 128, 128)
        '''
        # feed the inputs to the encoder
        encoder_output, convs = self.encoder(inputs)

        # feed the encoder output to the bottleneck
        bottle_neck = self.bottleneck(encoder_output)

        # feed the bottleneck and encoder block outputs to the decoder
        return self.decoder(bottle_neck, convs)


def unet(device):
    '''
    Defines the UNet and moves it to the chosen device.

    Args:
      device (torch.device) -- device the model is moved to

    Returns:
      nn.Module -- the assembled UNet
    '''
    return UNet(OUTPUT_CHANNELS).to(device)


# instantiate the model
model = unet(device)

# see the resulting model architecture
summary(model, input_size=(1, 3, 128, 128), device=device, depth=2)

## Configure and Train the model

Now, all that is left to do is to configure and train the model. The loss you will use is cross entropy. The reason is because the network is trying to assign each pixel a label, just like multi-class prediction. In the true segmentation mask, each pixel is either 0, 1 or 2. The network outputs three channels, one per class, and `nn.CrossEntropyLoss` is the recommended loss for such a scenario. It applies the softmax internally, which is why the decoder returns raw logits.

In [ ]:
# configure the optimizer and the loss for training
optimizer = torch.optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()

In [ ]:
def run_epoch(loader, model, loss_fn, optimizer, device, train, steps=None):
    '''
    Runs one pass over `loader`, training or evaluating.

    Args:
      loader (DataLoader) -- yields (images, masks) batches
      model (nn.Module) -- the UNet being trained or evaluated
      loss_fn (callable) -- loss applied to (logits, masks)
      optimizer (Optimizer) -- updates weights; only used when train is True
      device (torch.device) -- device the batches are moved to
      train (bool) -- True updates the weights, False only measures
      steps (int) -- stop after this many batches, or None for the whole loader

    Returns:
      (float, float) -- mean loss and mean per-pixel accuracy
    '''
    model.train(train)
    total_loss, correct, count = 0.0, 0.0, 0
    with torch.set_grad_enabled(train):
        for step, (images, masks) in enumerate(loader):
            if steps is not None and step >= steps:
                break
            images, masks = images.to(device), masks.to(device)
            logits = model(images)                       # shape: (N, 3, 128, 128)
            loss = loss_fn(logits, masks)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(images)
            correct += (logits.argmax(1) == masks).float().mean().item() * len(images)
            count += len(images)
    return total_loss / count, correct / count


# configure the training parameters and train the model
EPOCHS = 10
VAL_SUBSPLITS = 5
VALIDATION_STEPS = len(test) // BATCH_SIZE // VAL_SUBSPLITS

history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}

# this will take a while to run
for epoch in range(EPOCHS):
    train_loss, train_acc = run_epoch(train_dataset, model, loss_fn, optimizer, device, train=True)
    val_loss, val_acc = run_epoch(test_dataset, model, loss_fn, optimizer, device, train=False,
                                  steps=VALIDATION_STEPS)
    history['loss'].append(train_loss); history['accuracy'].append(train_acc)
    history['val_loss'].append(val_loss); history['val_accuracy'].append(val_acc)
    print(f"Epoch {epoch + 1}/{EPOCHS} - loss: {train_loss:.4f} - accuracy: {train_acc:.4f} "
          f"- val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}")

You can plot the train and validation loss to see how the training went. This should show generally decreasing values per epoch.

In [ ]:
# Plot the training and validation loss
plot_metrics(history, "loss", title="Training vs Validation Loss", ylim=1)

## Make predictions

The model is now ready to make some predictions. You will use the test dataset you prepared earlier to feed input images that the model has not seen before. The utilities below will help in processing the test dataset and model predictions.

In [ ]:
# Prediction Utilities

def get_test_image_and_annotation_arrays(test_dataset):
    '''
    Unpacks the test dataset and returns the input images and segmentation masks.

    Args:
      test_dataset (DataLoader) -- batches of the test split

    Returns:
      (array, array) -- images of shape (N, 3, 128, 128) and label maps of shape (N, 128, 128)
    '''
    images, y_true_segments = [], []
    for image, annotation in test_dataset:
        images.append(image)
        y_true_segments.append(annotation)

    return torch.cat(images).numpy(), torch.cat(y_true_segments).numpy()


def create_mask(pred_mask):
    '''
    Creates the segmentation mask by taking the channel with the highest score.

    Remember that the UNet outputs 3 channels. For each pixel, the prediction is the
    channel with the highest score.

    Args:
      pred_mask (tensor) -- model output, shape (N, 3, H, W)

    Returns:
      array -- label map of the first image, shape (H, W)
    '''
    return pred_mask.argmax(dim=1)[0].cpu().numpy()


def make_predictions(image, model, device):
    '''
    Feeds a single image to the model and returns the predicted mask.

    Args:
      image (array) -- one preprocessed image, shape (3, 128, 128)
      model (nn.Module) -- the trained UNet
      device (torch.device) -- device the model runs on

    Returns:
      array -- predicted label map, shape (128, 128)
    '''
    model.eval()
    batch = torch.as_tensor(image).unsqueeze(0).to(device)   # shape: (1, 3, 128, 128)
    with torch.no_grad():
        pred_mask = model(batch)

    return create_mask(pred_mask)

### Compute class wise metrics

Like the previous lab, you will also want to compute the IOU and Dice Score. This is the same function you used previously.

In [ ]:
def class_wise_metrics(y_true, y_pred):
    '''
    Computes the class-wise IOU and Dice Score.

    Args:
      y_true (array) -- ground truth label maps
      y_pred (array) -- predicted label maps of the same shape

    Returns:
      (list, list) -- IOU and Dice score, one entry per class
    '''
    class_wise_iou = []
    class_wise_dice_score = []

    smoothening_factor = 0.00001
    for i in range(3):

        intersection = np.sum((y_pred == i) * (y_true == i))
        y_true_area = np.sum((y_true == i))
        y_pred_area = np.sum((y_pred == i))
        combined_area = y_true_area + y_pred_area

        iou = (intersection + smoothening_factor) / (combined_area - intersection + smoothening_factor)
        class_wise_iou.append(iou)

        dice_score = 2 * ((intersection + smoothening_factor) / (combined_area + smoothening_factor))
        class_wise_dice_score.append(dice_score)

    return class_wise_iou, class_wise_dice_score

With all the utilities defined, you can now proceed to showing the metrics and feeding test images.

In [ ]:
# Setup the ground truth and predictions.

# get the ground truth from the test set
y_true_images, y_true_segments = get_test_image_and_annotation_arrays(test_dataset)

# feed the test set to the model to get the predicted masks
model.eval()
results = []
with torch.no_grad():
    for images, _ in test_dataset:
        results.append(model(images.to(device)).argmax(dim=1).cpu())
results = torch.cat(results).numpy()

print("predictions:", results.shape)

In [ ]:
# compute the class wise metrics
cls_wise_iou, cls_wise_dice_score = class_wise_metrics(y_true_segments, results)

In [ ]:
# show the IOU for each class
for idx, iou in enumerate(cls_wise_iou):
    spaces = ' ' * (10 - len(class_names[idx]) + 2)
    print("{}{}{} ".format(class_names[idx], spaces, iou))

In [ ]:
# show the Dice Score for each class
for idx, dice_score in enumerate(cls_wise_dice_score):
    spaces = ' ' * (10 - len(class_names[idx]) + 2)
    print("{}{}{} ".format(class_names[idx], spaces, dice_score))

### Show Predictions

In [ ]:
# Please input a number between 0 and the size of the test set to pick an image
integer_slider = 3646
integer_slider = min(integer_slider, len(y_true_images) - 1)

# Get the prediction mask
y_pred_mask = make_predictions(y_true_images[integer_slider], model, device)

# Compute the class wise metrics
iou, dice_score = class_wise_metrics(y_true_segments[integer_slider], y_pred_mask)

# Overlay the metrics with the images
display_with_metrics([y_true_images[integer_slider], y_pred_mask, y_true_segments[integer_slider]],
                     iou, dice_score)

**That's all for this lab! In the next section, you will learn about another type of image segmentation model: Mask R-CNN for instance segmentation!**